## Setup
Run **Pipeline_Execution.ipynb C0 → C11** at least once before this notebook.  
This notebook reads from `pipeline_output/` and writes `pipeline_output/graph.json`.  
**Select kernel:** `Python (Spradley)` before running.

In [ ]:
# ── G0: Config ────────────────────────────────────────────────────────────────
# All thresholds are relative (share of N speakers) and transfer across datasets.
# Edit here before running. graph_core.py defaults are shown as comments.

import os
from dotenv import load_dotenv
import graph_core
import pipeline_core

# override=True so re-running this cell always picks up the latest keys.env
load_dotenv(pipeline_core.KEYS_ENV, override=True)
_key = os.getenv("ANTHROPIC_API_KEY")
print(f"API key loaded: {'yes' if _key else 'NO -- check keys.env'}")

# ── Dataset identity (scopes the Neo4j clean step) ────────────────────────────
graph_core.CONFIG["DATASET_ID"]          = "the-office-2"

# ── Relationship mode ─────────────────────────────────────────────────────────
# "per_interview": one LLM call per interview (more grounded, N calls total)
# "global":        one LLM call over cluster summaries (faster, less grounded)
graph_core.CONFIG["RELATIONSHIP_MODE"]   = "per_interview"

# ── Edge thresholds (share of N speakers, strict greater than) ────────────────
graph_core.CONFIG["CO_OCCURRENCE_MIN_SHARE"]  = 0.20
graph_core.CONFIG["MIN_EDGE_SPEAKER_SHARE"]   = 0.20

# ── Relationship vocabulary and caps ──────────────────────────────────────────
graph_core.CONFIG["RELATIONSHIP_VOCAB_CAP"]   = 8    # distinct canonical types
graph_core.CONFIG["TYPED_EDGES_TOTAL_CAP"]    = 12   # total surviving typed edges
graph_core.CONFIG["MAX_EDGES_PER_PAIR"]       = 2    # allow 2 types per pair (shows ambiguity)

# ── Entity extraction (G2b) ───────────────────────────────────────────────────
graph_core.CONFIG["MIN_ENTITY_NODES"]         = 4    # LLM picks between min and max
graph_core.CONFIG["MAX_ENTITY_NODES"]         = 7
graph_core.CONFIG["MIN_ENTITY_SPEAKER_COUNT"] = 2    # ignore entities < 2 speakers

# ── Neo4j upload options ──────────────────────────────────────────────────────
# CO_OCCURS_WITH edges are kept in graph.json for reference but not uploaded --
# they dominate the graph visually without adding analytical value.
graph_core.CONFIG["UPLOAD_CO_EDGES"]          = False

# ── LLM settings for graph calls (low temperature for consistency) ────────────
graph_core.CONFIG["LLM_TEMPERATURE"]          = 0.1

print("\nGraph config active:")
for k, v in graph_core.CONFIG.items():
    print(f"  {k}: {v}")
print(f"\nSeed relationship types: {graph_core.SEED_RELATIONSHIP_TYPES}")

In [ ]:
# ── G1: Load pipeline outputs + build nodes (deterministic) ───────────────────
# Reads clusters.json, lineage.json, interview_store.json, db.json,
# experiments.json, run_meta.json from pipeline_output/.
#
# Note: experiments.json must contain 'source_cluster' on each experiment.
# If you see a ValueError here, re-run cells C9 and C11 in
# Pipeline_Execution.ipynb first (C9 was updated to add that field).

data = graph_core.load_pipeline_outputs()

clusters        = data["clusters"]
lineage         = data["lineage"]
interview_store = data["interview_store"]
db              = data["db"]
experiments     = data["experiments"]
run_meta        = data["run_meta"]

N = graph_core.get_n(run_meta, interview_store)

cluster_nodes             = graph_core.build_cluster_nodes(clusters, lineage, N)
exp_nodes, addresses_edges = graph_core.build_experiment_nodes(experiments, clusters)

cluster_nodes_by_name = {n["name"]: n for n in cluster_nodes}

print(f"N (speakers): {N}")
print(f"Cluster nodes: {len(cluster_nodes)}")
for n in cluster_nodes:
    print(f"  [{n['category'][:4]}]  {n['name']}")
    print(f"           support={n['support_count']}  speakers={n['speaker_count']}  spread={n['spread']}")
print(f"\nExperiment nodes: {len(exp_nodes)}")
print(f"ADDRESSES edges:  {len(addresses_edges)}")

In [ ]:
# ── G2: Co-occurrence edges (deterministic) ───────────────────────────────────
# Two clusters co-occur when they both have at least one L2 code from the same
# interview. Weight = share of speakers in whose interviews both appear.
# Only edges with weight > CO_OCCURRENCE_MIN_SHARE are kept (strict).

interview_cluster_map = graph_core.build_interview_cluster_map(lineage)
co_edges              = graph_core.build_co_occurrence_edges(interview_cluster_map, N)

print(f"Interviews mapped: {len(interview_cluster_map)}")
print(f"Co-occurrence edges kept (weight > {graph_core.CONFIG['CO_OCCURRENCE_MIN_SHARE']:.0%}): {len(co_edges)}\n")
for e in co_edges:
    print(f"  {e['a']}")
    print(f"  <-> {e['b']}")
    print(f"  weight={e['weight']}  persons={e['person_count']}\n")

In [ ]:
# ── G2b: Entity extraction (LLM) ──────────────────────────────────────────────
# One LLM call over all Q&A evidence identifies between MIN_ENTITY_NODES and
# MAX_ENTITY_NODES analytically significant entities (roles, processes, tools,
# org units). The LLM freely proposes typed relationships to clusters,
# experiments, and other entities -- up to MAX_EDGES_PER_PAIR per pair to
# capture ambiguity (e.g. a manager who both causes stress AND enables coping).

entity_nodes, entity_edges = graph_core.extract_entity_nodes(
    db, clusters, lineage, N, exp_nodes=exp_nodes
)

print(f"Entity nodes: {len(entity_nodes)}")
for e in entity_nodes:
    print(f"  [{e['type']}]  {e['name']}  "
          f"(speakers={e['speaker_count']}, spread={e['spread']})")
    for cid in e.get("connected_clusters", []):
        print(f"    -> {cid}")

print(f"\nEntity edges: {len(entity_edges)}")
for ee in entity_edges:
    print(f"  {ee['source_id']}  --[{ee['type']}]-->  "
          f"{ee['target_id']}  [{ee['target_type']}]  "
          f"(speakers={ee['speaker_count']})")

In [ ]:
# ── G3: Per-interview typed proposals (LLM) ───────────────────────────────────
# For each interview with >=2 clusters: one LLM call proposes directed typed
# relationships grounded in that employee's Q&A evidence.
# Set RELATIONSHIP_MODE="global" in G0 for a single cheaper LLM call instead.

proposals = graph_core.run_g3_proposals(
    interview_cluster_map,
    cluster_nodes_by_name,
    lineage,
    db,
    co_edges=co_edges,
)

print(f"Total proposals: {len(proposals)}")
raw_types = sorted({p['type_raw'] for p in proposals})
print(f"Raw relationship types ({len(raw_types)}): {raw_types}")
print("\nSample proposals:")
for p in proposals[:5]:
    print(f"  {p['source_cluster']}")
    print(f"  --[{p['type_raw']}]--> {p['target_cluster']}")
    print(f"  {p['rationale'][:90]}\n")

In [ ]:
# ── G4: Consolidate (LLM rename + deterministic count) ───────────────────────
# (a) One LLM call merges synonym type labels into canonical types
#     (capped at RELATIONSHIP_VOCAB_CAP).
# (b) Proposals are grouped by (source, target, canonical_type).
#     Weight = distinct persons / N.

type_mapping = graph_core.canonicalize_types(proposals)
groups       = graph_core.count_groups(proposals, type_mapping, N)

canonical_types = sorted(set(type_mapping.values()))
print(f"Canonical types ({len(canonical_types)}): {canonical_types}")
print(f"\nGroups before caps: {len(groups)}")
groups_sorted = sorted(groups, key=lambda g: g['weight'], reverse=True)
for g in groups_sorted[:8]:
    print(f"  {g['source_cluster']}  --[{g['type']}]-->  {g['target_cluster']}")
    print(f"  weight={g['weight']}  persons={g['person_count']}\n")

In [ ]:
# ── G5: Finalize edges + assemble + save graph.json ───────────────────────────
# Applies caps (MIN_EDGE_SPEAKER_SHARE, per-pair dedup up to MAX_EDGES_PER_PAIR,
# TYPED_EDGES_TOTAL_CAP), then one LLM call writes a one-sentence context +
# is_paradox for each edge. Assembles graph.json and writes it to pipeline_output/.
#
# Requires entity_nodes and entity_edges from G2b.

typed_edges = graph_core.finalize_typed_edges(groups)

print(f"Typed edges after caps: {len(typed_edges)}")
for e in typed_edges:
    paradox = "[paradox]" if e.get('is_paradox') else ""
    print(f"  {e['source_cluster']}  --[{e['type']}]--> {e['target_cluster']}  "
          f"w={e['weight']} {paradox}")
    if e.get('context'):
        print(f"  '{e['context']}'")
    print()

# Hub score: fraction of typed edges each cluster appears in.
# High hub_score + needs_work category = highest-priority intervention point.
cluster_nodes = graph_core.flag_hub_scores(cluster_nodes, typed_edges)

graph = graph_core.assemble_graph(
    cluster_nodes, exp_nodes, co_edges, typed_edges, addresses_edges,
    entity_nodes, entity_edges,
)
graph_core.save_graph(graph)

### G6 -- Neo4j setup

Before running G6 you need a free Neo4j Aura instance and its connection credentials.

**Create an instance:**
1. Go to [console.neo4j.io](https://console.neo4j.io) and sign in (free account).
2. Create a new **AuraDB Free** instance.
3. At creation time Aura shows you the generated password **once** -- copy it now.
4. The instance detail page shows the Connection URI (`neo4j+s://...`).
5. Default username is `neo4j`.

**Add to `keys.env`** (project root, never committed):
```
NEO4J_URI=neo4j+s://<instance-id>.databases.neo4j.io
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=<your-generated-password>
```

**Install the driver** (once, in the Spradley venv):
```bash
pip install neo4j
```

**APOC:** Aura Free includes APOC by default -- dynamic relationship types work out of the box.
If APOC is absent, G6 falls back to `RELATES_TO` edges with a `type` property.

**After a kernel restart:** run G0 only, then run G6. `graph.json` is loaded from disk automatically.

---

**Verify the graph loaded correctly** (run in Neo4j Browser or Bloom):
```cypher
// Show full graph -- all node types and relationships
MATCH (n {dataset: "the-office-2"})
OPTIONAL MATCH (n)-[r]-(m {dataset: "the-office-2"})
RETURN n, r, m
```

**Size nodes in Bloom:**
- Cluster nodes: size by `hub_score` or `spread`
- Entity nodes: size by `speaker_count`
- Relationship thickness: size by `weight` or `speaker_count`

In [ ]:
# ── G6: Load into Neo4j ───────────────────────────────────────────────────────
# Wipes the current DATASET_ID scope in Neo4j, then MERGEs all nodes and edges.
# Running this cell twice for the same dataset is safe (MERGE is idempotent).
#
# After a kernel restart: run G0 (env + config), then run this cell only.
# graph.json is loaded from disk automatically if graph is not in memory.

import os
import socket
from dotenv import load_dotenv

load_dotenv(pipeline_core.KEYS_ENV, override=True)

missing = [k for k in ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"] if not os.getenv(k)]
if missing:
    raise EnvironmentError(
        f"Missing keys in keys.env: {missing}\n"
        "See the G6 setup note above for instructions."
    )

# ── Auto-load graph from disk if not in memory ────────────────────────────────
if "graph" not in dir() or graph is None:
    graph = graph_core._load_json("graph.json")
    print(f"Loaded graph.json from disk  ({len(graph['nodes'])} clusters, "
          f"{len(graph['co_edges'])} co-edges, {len(graph['typed_edges'])} typed edges)\n")

uri  = os.environ["NEO4J_URI"]
user = os.environ["NEO4J_USERNAME"]
print(f"Connecting to: {uri}")
print(f"Dataset scope: {graph['dataset']}\n")

# ── Port reachability check ───────────────────────────────────────────────────
import urllib.parse
_parsed = urllib.parse.urlparse(uri)
_host   = _parsed.hostname
_port   = _parsed.port or 7687
try:
    s = socket.create_connection((_host, _port), timeout=5)
    s.close()
    print(f"Port check: {_host}:{_port} reachable\n")
except OSError as e:
    print(f"Port check FAILED: {_host}:{_port} -- {e}")
    print(
        "\nMost likely cause: corporate firewall blocks outbound port 7687 (Bolt protocol).\n"
        "Options:\n"
        "  1. Connect via VPN / personal hotspot and re-run.\n"
        "  2. Use Neo4j Desktop locally (bolt://localhost:7687) -- no firewall issue.\n"
        "  3. Ask IT if port 7687 outbound is permitted.\n"
    )
    raise SystemExit("Stopping -- fix network connectivity first.")

graph_core.load_to_neo4j(graph)